# 🎓 Agent 65: Fine-Tuning Llama 3.2 3B for University Student Helpdesk
This notebook fine-tunes **Llama 3.2 3B Instruct** using **Unsloth & QLoRA** on a **Free Google Colab T4 GPU**.
- ⏱️ **Training Time**: ~15–20 minutes
- 💾 **VRAM Usage**: < 6 GB (fits easily on free Colab T4 16GB)
- 📦 **Output**: Merged `agent65-llama3.2-3b-q4_k_m.gguf` ready for local **Ollama** deployment with **zero API keys**!

### Step 1: Install High-Speed Fine-Tuning Stack

In [ ]:
# Install unsloth, unsloth_zoo and dependencies with full resolution
!pip install --upgrade pip
!pip install unsloth unsloth_zoo
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets triton

### Step 2: Load Base Llama 3.2 3B with 4-bit Quantization

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto detection (Float16 or Bfloat16)
load_in_4bit = True # 4bit quantization saves memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA Adapters for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Llama 3.2 3B model loaded with LoRA adapters!")

### Step 3: Upload `final_train_dataset.jsonl`

In [ ]:
from google.colab import files
import os
import glob

print("Upload `final_train_dataset.jsonl` from your backend/data folder:")
uploaded = files.upload()

# Auto-detect uploaded jsonl file
jsonl_files = [f for f in glob.glob("*.jsonl") if "val" not in f]
target_file = jsonl_files[0] if jsonl_files else "final_train_dataset.jsonl"
print(f"[*] Ingesting training dataset: {target_file}")

from datasets import load_dataset
dataset = load_dataset("json", data_files=target_file, split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Successfully formatted {len(dataset)} student helpdesk dialogues!")

### Step 4: Fine-Tune with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # ~2-3 epochs over the dataset
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Starting fast training on GPU...")
trainer_stats = trainer.train()
print("🎉 Training complete!")

### Step 5: Export to GGUF for 100% Offline Local Ollama Deployment

In [ ]:
# Export directly to GGUF with Q4_K_M quantization (compact, fast CPU inference)
model.save_pretrained_gguf("agent65_model", tokenizer, quantization_method = "q4_k_m")

# Download the GGUF to your local machine
import glob
from google.colab import files
gguf_files = glob.glob("agent65_model*/**/*.gguf", recursive=True) + glob.glob("*.gguf")
print(f"Found GGUF models: {gguf_files}")
if gguf_files:
    print(f"Triggering download for: {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("Check the left sidebar files tab under agent65_model_gguf!")

### Step 6: Deploy Locally in Ollama (On your Windows PC)
Once downloaded to your computer:
1. Move the `.gguf` file to your `StudentHelpdesk/backend/models/` folder.
2. In `backend/models/Modelfile.agent65`, point to the gguf:
   ```dockerfile
   FROM ./Llama-3.2-3B-Instruct.Q4_K_M.gguf
   ```
3. Run in terminal:
   ```powershell
   ollama create agent65 -f backend/models/Modelfile.agent65
   ```
4. Agent 65 is now running your fine-tuned model 100% locally with zero API keys!